# Product quantity and price

**Question:** How does price relate to recorded quantity for comparable products? The starting population is raw rows with `Quantity > 0` and `UnitPrice > 0`. This is a provisional recorded-sales population, not verified demand or completed sales.

In [1]:
from pathlib import Path as Path02
import sys as sys02
import pandas as pandas02
from IPython.display import display as display02

project_root_candidates_02 = (Path02.cwd(), *Path02.cwd().parents)
project_root_02 = next(
    (candidate for candidate in project_root_candidates_02 if (candidate / "src" / "clean.py").is_file()),
    None,
)
if project_root_02 is None:
    raise FileNotFoundError("Could not locate the project root from the current directory.")
if str(project_root_02) not in sys02.path:
    sys02.path.insert(0, str(project_root_02))
from src.clean import load_raw as load_raw_02, match_cancelled_orders as match_cancelled_orders_02

relative_data_path_02 = Path02("data") / "raw" / "Online Retail.xlsx"
data_path_02 = project_root_02 / relative_data_path_02
retail_02 = load_raw_02(data_path_02)
retail_snapshot_02 = retail_02.copy(deep=True)
print(f"Workbook: {relative_data_path_02}")
print(f"Raw rows and columns: {retail_02.shape}")

Workbook: data\raw\Online Retail.xlsx
Raw rows and columns: (541909, 8)


## Product rankings

The two rankings use only the starting population. Ties are ordered by `StockCode` for reproducibility.

In [2]:
positive_population_02 = retail_02.loc[
    retail_02["Quantity"].gt(0) & retail_02["UnitPrice"].gt(0)
].copy()
quantity_metrics_02 = positive_population_02.groupby("StockCode", as_index=False).agg(
    total_quantity=("Quantity", "sum")
)
quantity_ranking_02 = quantity_metrics_02.sort_values(
    ["total_quantity", "StockCode"], ascending=[False, True], kind="mergesort"
).reset_index(drop=True)
top20_quantity_02 = quantity_ranking_02.head(20)
print("Top 20 products by total positive recorded quantity:")
display02(top20_quantity_02)

Top 20 products by total positive recorded quantity:


,StockCode,total_quantity
0,23843,80995
1,23166,78033
2,22197,56921
3,84077,55047
4,85099B,48474
5,85123A,37952
6,84879,36461
7,21212,36419
8,23084,30788
9,22492,26633


In [3]:
invoice_metrics_02 = positive_population_02.groupby("StockCode", as_index=False).agg(
    distinct_invoices=("InvoiceNo", "nunique")
)
invoice_ranking_02 = invoice_metrics_02.sort_values(
    ["distinct_invoices", "StockCode"], ascending=[False, True], kind="mergesort"
).reset_index(drop=True)
top20_invoices_02 = invoice_ranking_02.head(20)
print("Top 20 products by distinct positive-population invoices:")
display02(top20_invoices_02)

Top 20 products by distinct positive-population invoices:


,StockCode,distinct_invoices
0,85123A,2265
1,85099B,2089
2,22423,1988
3,47566,1685
4,20725,1565
5,84879,1455
6,22197,1392
7,22720,1385
8,21212,1320
9,22383,1284


## Why StockCode 23843 looks suspicious

Inspect every raw row for this code, including its invoice number, quantity, customer, price, and date.

In [4]:
raw_rows_23843_02 = retail_02.loc[
    retail_02["StockCode"].astype("string").str.strip().eq("23843")
].copy()
display02(raw_rows_23843_02)
assert set(raw_rows_23843_02["InvoiceNo"].astype(str)) == {"581483", "C581484"}
assert set(raw_rows_23843_02["Quantity"]) == {80995, -80995}
assert raw_rows_23843_02["CustomerID"].nunique() == 1
print("The raw rows show +80,995 on 581483 and -80,995 on C581484 for the same customer.")

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
540421,581483,23843,"PAPER CRAFT , LITTLE BIRDIE",80995,2011-12-09 09:15:00,2.08,16446.0,United Kingdom
540422,C581484,23843,"PAPER CRAFT , LITTLE BIRDIE",-80995,2011-12-09 09:27:00,2.08,16446.0,United Kingdom


The raw rows show +80,995 on 581483 and -80,995 on C581484 for the same customer.


## StockCode 23166: largest line and cancellation

Measure the largest positive-population line against this product's total, then show the exact matched cancellation pair.

In [5]:
stock_codes_23166_02 = retail_02["StockCode"].astype("string").str.strip()
sales_rows_23166_02 = positive_population_02.loc[stock_codes_23166_02.eq("23166")]
total_quantity_23166_02 = int(sales_rows_23166_02["Quantity"].sum())
largest_line_23166_02 = sales_rows_23166_02.sort_values(
    ["Quantity", "InvoiceNo"], ascending=[False, True], kind="mergesort"
).iloc[0]
largest_share_23166_02 = largest_line_23166_02["Quantity"] / total_quantity_23166_02
raw_for_matching_02 = retail_02.copy()
raw_for_matching_02["StockCode"] = raw_for_matching_02["StockCode"].astype(str).str.strip()
matched_positive_02, matched_cancellation_02 = match_cancelled_orders_02(raw_for_matching_02)
matched_pairs_23166_02 = pandas02.DataFrame({
    "InvoiceNo": raw_for_matching_02.loc[matched_positive_02, "InvoiceNo"].to_numpy(),
    "C-InvoiceNo": raw_for_matching_02.loc[matched_cancellation_02, "InvoiceNo"].to_numpy(),
    "StockCode": raw_for_matching_02.loc[matched_positive_02, "StockCode"].to_numpy(),
    "Quantity": raw_for_matching_02.loc[matched_positive_02, "Quantity"].to_numpy(),
    "CustomerID": raw_for_matching_02.loc[matched_positive_02, "CustomerID"].to_numpy(),
})
matched_pairs_23166_02 = matched_pairs_23166_02.loc[
    matched_pairs_23166_02["StockCode"].eq("23166")
]
print(f"23166 positive-population total: {total_quantity_23166_02:,}")
print(
    f"Largest line: {int(largest_line_23166_02['Quantity']):,} of "
    f"{total_quantity_23166_02:,} ({largest_share_23166_02:.1%}); "
    f"InvoiceNo={largest_line_23166_02['InvoiceNo']}"
)
display02(matched_pairs_23166_02.reset_index(drop=True))
assert total_quantity_23166_02 == 78033
assert int(largest_line_23166_02["Quantity"]) == 74215
assert len(matched_pairs_23166_02) >= 1

23166 positive-population total: 78,033
Largest line: 74,215 of 78,033 (95.1%); InvoiceNo=541431


,InvoiceNo,C-InvoiceNo,StockCode,Quantity,CustomerID
0,541431,C541433,23166,74215,12346.0
1,553607,C557508,23166,240,16684.0
2,560491,C560540,23166,1,12415.0
3,562374,C562375,23166,12,14911.0
4,570467,C570867,23166,12,12607.0


## Ranking sensitivity

Compare (a) the original positive-quantity, positive-price ranking, (b) that ranking after removing each product's single largest positive line, and (c) net signed quantity using only rows with `UnitPrice > 0`. Zero-price rows are excluded from all three comparisons.

In [6]:
original_sensitivity_totals_02 = positive_population_02.groupby("StockCode")["Quantity"].sum()
ranking_a_02 = original_sensitivity_totals_02.rename("quantity").reset_index().sort_values(
    ["quantity", "StockCode"], ascending=[False, True], kind="mergesort"
).reset_index(drop=True)
largest_line_by_product_02 = positive_population_02.groupby("StockCode")["Quantity"].max()
without_largest_totals_02 = (original_sensitivity_totals_02 - largest_line_by_product_02).reindex(
    original_sensitivity_totals_02.index, fill_value=0
)
ranking_b_02 = without_largest_totals_02.rename("quantity").reset_index().sort_values(
    ["quantity", "StockCode"], ascending=[False, True], kind="mergesort"
).reset_index(drop=True)
positive_price_signed_rows_02 = retail_02.loc[retail_02["UnitPrice"].gt(0)]
net_signed_totals_02 = positive_price_signed_rows_02.groupby("StockCode")["Quantity"].sum()
ranking_c_02 = net_signed_totals_02.rename("quantity").reset_index().sort_values(
    ["quantity", "StockCode"], ascending=[False, True], kind="mergesort"
).reset_index(drop=True)
original_top20_codes_02 = set(ranking_a_02.head(20)["StockCode"])
for ranking_label_02, ranking_frame_02 in [
    ("(a) Original", ranking_a_02),
    ("(b) Without each product's largest line", ranking_b_02),
    ("(c) Net signed quantity; UnitPrice > 0 only", ranking_c_02),
]:
    ranking_frame_02["rank"] = range(1, len(ranking_frame_02) + 1)
    current_top20_codes_02 = set(ranking_frame_02.head(20)["StockCode"])
    two_code_ranks_02 = ranking_frame_02.loc[
        ranking_frame_02["StockCode"].astype("string").str.strip().isin(["23843", "23166"]),
        ["rank", "StockCode", "quantity"],
    ]
    print(ranking_label_02)
    display02(ranking_frame_02.head(20))
    display02(two_code_ranks_02.reset_index(drop=True))
    print(f"Original top-20 products remaining: {len(original_top20_codes_02.intersection(current_top20_codes_02))} of 20")
quantity_23843_02 = int(ranking_b_02.loc[ranking_b_02["StockCode"].astype("string").str.strip().eq("23843"), "quantity"].iloc[0])
rank_23843_02 = int(ranking_b_02.loc[ranking_b_02["StockCode"].astype("string").str.strip().eq("23843"), "rank"].iloc[0])
tied_products_23843_02 = int(ranking_b_02["quantity"].eq(quantity_23843_02).sum())
print(f"23843 quantity: {quantity_23843_02}; rank: {rank_23843_02}; products tied at that quantity: {tied_products_23843_02}")
assert quantity_23843_02 == 0
assert int(ranking_b_02.loc[ranking_b_02["StockCode"].astype("string").str.strip().eq("23166"), "quantity"].iloc[0]) == 3818
assert len(original_top20_codes_02.intersection(set(ranking_b_02.head(20)["StockCode"]))) == 18

(a) Original


,StockCode,quantity,rank
0,23843,80995,1
1,23166,78033,2
2,22197,56921,3
3,84077,55047,4
4,85099B,48474,5
5,85123A,37952,6
6,84879,36461,7
7,21212,36419,8
8,23084,30788,9
9,22492,26633,10


,rank,StockCode,quantity
0,1,23843,80995
1,2,23166,78033


Original top-20 products remaining: 20 of 20
(b) Without each product's largest line


,StockCode,quantity,rank
0,22197,52621,1
1,84077,50247,2
2,85099B,47274,3
3,85123A,36022,4
4,21212,34979,5
5,84879,33581,6
6,23084,28388,7
7,22492,25239,8
8,22616,24839,9
9,22178,23344,10


,rank,StockCode,quantity
0,319,23166,3818
1,3721,23843,0


Original top-20 products remaining: 18 of 20
(c) Net signed quantity; UnitPrice > 0 only


,StockCode,quantity,rank
0,22197,56450,1
1,84077,53847,2
2,85099B,47359,3
3,84879,36381,4
4,21212,36039,5
5,85123A,35374,6
6,23084,30680,7
7,22492,26437,8
8,22616,26111,9
9,21977,24753,10


,rank,StockCode,quantity
0,377,23166,3539
1,3800,23843,0


Original top-20 products remaining: 18 of 20
23843 quantity: 0; rank: 3721; products tied at that quantity: 128


## StockCode 84347 and file boundary

Show the recorded -9,360 return, whether any positive line for the same product predates it in this file, the file start, and the number of products whose first negative row precedes their first positive row.

In [7]:
raw_rows_84347_02 = retail_02.loc[
    retail_02["StockCode"].astype("string").str.strip().eq("84347")
].copy()
negative_rows_84347_02 = raw_rows_84347_02.loc[raw_rows_84347_02["Quantity"].lt(0)]
assert len(negative_rows_84347_02.loc[negative_rows_84347_02["Quantity"].eq(-9360)]) == 1
return_row_84347_02 = negative_rows_84347_02.loc[
    negative_rows_84347_02["Quantity"].eq(-9360)
].iloc[0]
positive_rows_84347_02 = raw_rows_84347_02.loc[
    raw_rows_84347_02["Quantity"].gt(0) & raw_rows_84347_02["UnitPrice"].gt(0)
]
return_rate_84347_02 = abs(return_row_84347_02["Quantity"]) / positive_rows_84347_02["Quantity"].sum()
earlier_sales_84347_02 = positive_rows_84347_02.loc[
    positive_rows_84347_02["InvoiceDate"].lt(return_row_84347_02["InvoiceDate"])
]
earlier_same_customer_84347_02 = earlier_sales_84347_02.loc[
    earlier_sales_84347_02["CustomerID"].eq(return_row_84347_02["CustomerID"])
]
first_negative_by_code_02 = retail_02.loc[retail_02["Quantity"].lt(0)].groupby("StockCode")["InvoiceDate"].min()
first_positive_by_code_02 = retail_02.loc[retail_02["Quantity"].gt(0)].groupby("StockCode")["InvoiceDate"].min()
first_date_comparison_02 = pandas02.concat(
    [first_negative_by_code_02.rename("first_negative"), first_positive_by_code_02.rename("first_positive")],
    axis=1,
)
negative_before_positive_count_02 = int(
    (first_date_comparison_02["first_negative"].notna()
     & first_date_comparison_02["first_positive"].notna()
     & first_date_comparison_02["first_negative"].lt(first_date_comparison_02["first_positive"])).sum()
)
print(f"File starts: {retail_02['InvoiceDate'].min()}")
print("The single -9,360 return row:")
display02(negative_rows_84347_02.loc[negative_rows_84347_02["Quantity"].eq(-9360)])
print(
    f"Positive units: {int(positive_rows_84347_02['Quantity'].sum()):,}; "
    f"return units: 9,360; return / positive units: {return_rate_84347_02:.1%}"
)
print(
    f"Earlier positive 84347 rows in the file: {len(earlier_sales_84347_02)}; "
    f"earlier rows for this customer: {len(earlier_same_customer_84347_02)}"
)
display02(
    earlier_sales_84347_02[
        ["InvoiceNo", "Quantity", "CustomerID", "InvoiceDate"]
    ].reset_index(drop=True)
)
print(f"Products whose first negative row precedes their first positive row: {negative_before_positive_count_02:,}")
assert earlier_same_customer_84347_02.empty

File starts: 2010-12-01 08:26:00
The single -9,360 return row:


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
4287,C536757,84347,ROTATING SILVER ANGELS T-LIGHT HLDR,-9360,2010-12-02 14:23:00,0.03,15838.0,United Kingdom


Positive units: 9,476; return units: 9,360; return / positive units: 98.8%
Earlier positive 84347 rows in the file: 10; earlier rows for this customer: 0


,InvoiceNo,Quantity,CustomerID,InvoiceDate
0,536408,6,14307.0,2010-12-01 11:41:00
1,536446,12,15983.0,2010-12-01 12:15:00
2,536488,1,17897.0,2010-12-01 12:31:00
3,536488,1,17897.0,2010-12-01 12:31:00
4,536488,3,17897.0,2010-12-01 12:31:00
5,536527,6,12662.0,2010-12-01 13:04:00
6,536544,4,NaN,2010-12-01 14:32:00
7,536592,1,NaN,2010-12-01 17:06:00
8,536667,6,15260.0,2010-12-02 12:09:00
9,536749,1,17976.0,2010-12-02 13:49:00


Products whose first negative row precedes their first positive row: 75


In [8]:
assert retail_02.equals(retail_snapshot_02), "The source dataframe changed during this analysis."
print("Source snapshot check passed: retail_02.equals(retail_snapshot_02).")

Source snapshot check passed: retail_02.equals(retail_snapshot_02).

## Conclusions

**Observations:** the raw rows for 23843 contain equal positive and cancellation quantities for the same customer; 23166 has a 74,215-unit largest line out of 78,033 positive-price units; ranking positions change substantially when each product's largest line is removed or signed quantities are netted.

**Interpretations and limitations:** these patterns are consistent with exceptional/cancelled transactions, but recorded quantities alone do not establish why they were entered. A product ranking is sensitive to its population definition. Returns may be unmatched or refer to sales before the file begins, and positive-quantity rows are not verified demand. Price/quantity association is not causation.

For the exact-match cleaning rules and clean-data diagnostics, see the [clean foundation](./03_clean_foundation.ipynb).